# CUPED Variance Reduction — Demo

This notebook demonstrates how CUPED (Controlled-experiment Using Pre-Experiment Data)
reduces metric variance and increases experiment sensitivity.

**Key insight**: By exploiting a pre-experiment covariate correlated with the primary metric,
CUPED can halve the required sample size when ρ ≈ 0.7.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from src.stats.cuped import cuped_adjust, variance_reduction_factor
from src.stats.power import required_sample_size, minimum_detectable_effect

sns.set_theme(style='whitegrid', palette='muted')
rng = np.random.default_rng(42)

## 1. Synthetic Data Generation

We simulate an experiment where:
- The **covariate** X is pre-experiment revenue (correlated with post-experiment revenue)
- The **treatment** increases revenue by `true_effect`
- We generate a range of correlation strengths to illustrate CUPED's benefit

In [ ]:
N_PER_GROUP = 500
TRUE_EFFECT = 0.5        # absolute lift in revenue
BASELINE_MEAN = 10.0
NOISE_STD = 2.0

def generate_experiment(rho: float, n: int = N_PER_GROUP, seed: int = 0):
    """Generate synthetic experiment data with known correlation rho between X and Y."""
    rng = np.random.default_rng(seed)
    # X (pre-experiment covariate)
    x_ctrl = rng.normal(BASELINE_MEAN, NOISE_STD, n)
    x_trt  = rng.normal(BASELINE_MEAN, NOISE_STD, n)

    # Y = rho*X + sqrt(1-rho^2)*noise  →  Corr(Y,X) = rho
    noise_ctrl = rng.normal(0, 1, n)
    noise_trt  = rng.normal(0, 1, n)

    y_ctrl = rho * x_ctrl + np.sqrt(1 - rho**2) * noise_ctrl * NOISE_STD
    y_trt  = rho * x_trt  + np.sqrt(1 - rho**2) * noise_trt  * NOISE_STD + TRUE_EFFECT

    return x_ctrl, x_trt, y_ctrl, y_trt

# Quick sanity check with rho = 0.7
x_c, x_t, y_c, y_t = generate_experiment(rho=0.7)
print(f"Control mean: {y_c.mean():.3f}, Treatment mean: {y_t.mean():.3f}")
print(f"Observed effect: {y_t.mean() - y_c.mean():.3f} (true: {TRUE_EFFECT})")
print(f"Corr(Y, X) control: {np.corrcoef(y_c, x_c)[0,1]:.3f}")

## 2. CUPED Adjustment

In [ ]:
y_c_adj, y_t_adj = cuped_adjust(y_c, y_t, x_c, x_t)

print("=== Variance Comparison (rho=0.7) ===")
print(f"Original  control variance: {np.var(y_c, ddof=1):.4f}")
print(f"Adjusted  control variance: {np.var(y_c_adj, ddof=1):.4f}")
vrf = variance_reduction_factor(np.concatenate([y_c, y_t]), np.concatenate([x_c, x_t]))
print(f"Variance reduction factor:  {vrf:.3f}  (theoretical: {1 - 0.7**2:.3f})")

print("\n=== Effect Preservation ===")
print(f"Original effect:  {y_t.mean() - y_c.mean():.4f}")
print(f"Adjusted effect:  {y_t_adj.mean() - y_c_adj.mean():.4f}  (should match)")

## 3. Power Gain Across Correlation Strengths

In [ ]:
rhos = np.linspace(0, 0.95, 20)
results = []

for rho in rhos:
    x_c, x_t, y_c, y_t = generate_experiment(rho=rho, seed=int(rho * 1000))
    y_c_adj, y_t_adj = cuped_adjust(y_c, y_t, x_c, x_t)

    std_orig = np.std(np.concatenate([y_c, y_t]), ddof=1)
    std_adj  = np.std(np.concatenate([y_c_adj, y_t_adj]), ddof=1)

    n_orig = required_sample_size(TRUE_EFFECT, std_orig)
    n_adj  = required_sample_size(TRUE_EFFECT, std_adj)

    results.append({
        'rho': rho,
        'n_original': n_orig,
        'n_cuped': n_adj,
        'sample_reduction_pct': (1 - n_adj / n_orig) * 100,
        'vrf': 1 - rho**2,
    })

df_results = pd.DataFrame(results)
df_results.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Sample size comparison
ax = axes[0]
ax.plot(df_results['rho'], df_results['n_original'], label='Standard t-test', linewidth=2)
ax.plot(df_results['rho'], df_results['n_cuped'],    label='CUPED-adjusted', linewidth=2)
ax.set_xlabel('Correlation ρ(Y, X)', fontsize=12)
ax.set_ylabel('Required Sample Size (per group)', fontsize=12)
ax.set_title('Sample Size: Standard vs CUPED', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.4)

# Right: Sample size reduction
ax = axes[1]
ax.fill_between(df_results['rho'], df_results['sample_reduction_pct'], alpha=0.4)
ax.plot(df_results['rho'], df_results['sample_reduction_pct'], linewidth=2)
ax.axvline(0.7, color='red', linestyle='--', alpha=0.6, label='ρ=0.7 (~51% reduction)')
ax.set_xlabel('Correlation ρ(Y, X)', fontsize=12)
ax.set_ylabel('Sample Size Reduction (%)', fontsize=12)
ax.set_title('CUPED Sample Size Savings', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig('../docs/cuped_power_gain.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. A/B Test: CUPED vs Standard t-test

Running 1000 simulated experiments to compare Type I error and power.

In [ ]:
from scipy.stats import ttest_ind

N_SIMS = 1000
RHO = 0.7
ALPHA = 0.05

standard_pvals = []
cuped_pvals    = []

for i in range(N_SIMS):
    x_c, x_t, y_c, y_t = generate_experiment(rho=RHO, n=200, seed=i)
    y_c_adj, y_t_adj = cuped_adjust(y_c, y_t, x_c, x_t)

    _, p_std  = ttest_ind(y_t, y_c)
    _, p_cuped = ttest_ind(y_t_adj, y_c_adj)

    standard_pvals.append(p_std)
    cuped_pvals.append(p_cuped)

power_std   = np.mean(np.array(standard_pvals) < ALPHA)
power_cuped = np.mean(np.array(cuped_pvals) < ALPHA)

print(f"n per group = 200, ρ = {RHO}, true effect = {TRUE_EFFECT}")
print(f"Standard t-test power: {power_std:.3f}")
print(f"CUPED-adjusted power:  {power_cuped:.3f}")
print(f"Relative power gain:   {(power_cuped - power_std) / power_std:.1%}")

## 5. Summary

| ρ (covariate correlation) | Variance Reduction | Sample Size Reduction |
|---|---|---|
| 0.0 | 0% | 0% |
| 0.5 | 25% | 25% |
| 0.7 | 51% | ~51% |
| 0.9 | 81% | ~81% |

**Best covariate candidates**: previous period metric, user lifetime value, engagement score.